# Zero-shot first-error detection

4 deterministic examples per task type. Full decoded chat transcripts (including special tokens) are cached per prompt/model/config in `cache/llm_critic.sqlite`, alongside benchmark, task, and example ID. The prompt uses 1-based `Step i:` labels for every benchmark, with zero-based labels internally. Accuracy is exact first-error localization (unparseable = incorrect); F1/MCC measure binary error detection on parseable responses. Coverage and unparseable counts are reported. Single-sample zero-shot protocol, not directly comparable to the papers' few-shot or majority-vote results.

In [1]:
from llm_critic import (load_examples, sample_tasks, evaluate, statistics, random_answers)

examples = sample_tasks(load_examples(), size=4, seed=42)

In [2]:
MODEL = "Qwen/Qwen3.5-9B"

## Evaluate non-thinking

In [3]:
results = evaluate(examples, model_id=MODEL, thinking=False, batch_size=16)

2026-10-29 15:19:36 INFO llm_critic: Starting evaluation: model=Qwen/Qwen3.5-9B examples=48 cache=/sc/home/jan.berndt/better-nli/final/cache/llm_critic.sqlite cache_only=False


Evaluating:   0%|          | 0/48 [00:00<?, ?example/s]

2026-10-29 15:19:36 WARNING llm_critic: UNPARSEABLE MR-GSM8K/reversed/1a74e73f-4a2f-4cf2-ad17-1f2a1e6f85a9: missing FINAL line at end of response
NONE

2026-10-29 15:19:37 WARNING llm_critic: UNPARSEABLE BIG-Bench-Mistake/word_sorting/86: missing FINAL line at end of response
The solution contains a logical error in Step 4.

In Step 3, the second letters are identified as:
- "include": "n" (14)
- "interrupt": "n" (14)
- "item": "t" (20)
- "ionosphere": "o" (15)
- "idyll": "d" (4)

The correct alphabetical order based on these second letters is:
1. "idyll" (d, 4)
2. "include" and "interrupt" (both n, 14)
2026-10-29 15:19:37 WARNING llm_critic: UNPARSEABLE BIG-Bench-Mistake/word_sorting/243: missing FINAL line at end of response
The solution contains a logical error in Step 10.

In Step 9, the second letters of "planoconcave" ('l') and "parch" ('a') are correctly identified. Since 'a' comes before 'l', "parch" should come before "planoconcave".
However, in Step 10, the solution states: "

In [4]:
display(statistics(results))
random_answers(results, examples, seed=42)

,benchmark,task,n,errors,unparseable,coverage,accuracy,error_accuracy,no_error_accuracy,f1,mcc
0,MR-GSM8K,original,4,3,0,1.00,0.50,0.333333,1.000000,0.800000,0.577350
1,MR-GSM8K,POT,4,1,0,1.00,0.50,1.000000,0.333333,0.500000,0.333333
2,MR-GSM8K,reversed,4,1,1,0.75,0.50,1.000000,0.333333,0.666667,0.500000
3,BIG-Bench-Mistake,dyck_languages,4,3,0,1.00,0.00,0.000000,0.000000,0.857143,0.000000
4,BIG-Bench-Mistake,logical_deduction,4,4,0,1.00,0.75,0.750000,NaN,1.000000,0.000000
5,BIG-Bench-Mistake,multistep_arithmetic,4,4,0,1.00,0.00,0.000000,NaN,0.857143,0.000000
6,BIG-Bench-Mistake,tracking_shuffled_objects,4,4,0,1.00,0.50,0.500000,NaN,0.666667,0.000000
7,BIG-Bench-Mistake,word_sorting,4,4,3,0.25,0.00,0.000000,NaN,0.000000,0.000000
8,ProcessBench,gsm8k,4,2,0,1.00,1.00,1.000000,1.000000,1.000000,1.000000
9,ProcessBench,math,4,0,0,1.00,1.00,NaN,1.000000,0.000000,0.000000



=== MR-GSM8K / original / ffe7fcb1-7651-4385-af1b-43c4e84d6151 ===
Question:
On a certain day, the total cost of filling up 20 helium balloons was $900. Two days later, the cost of filling each balloon was increased by $20. How much did Bentley pay to fill 170 balloons if he went to fill them after the price increase?

Solution steps:
Step 1: On the first day, the cost of filling each balloon was $900 / 20 = $45.
Step 2: Two days later, the cost of filling each balloon increased by $20, so it became $45 + $20 = $65.
Step 3: Bentley filled 170 balloons, so he paid a total of $65 * 170 = $10,550.
Step 4: #### 10550
Step 5: The answer is: 10550

Correct answer: 11050
Ground truth first error step: Step 3
LLM judge's output:
FINAL: NONE

Is match: False


=== MR-GSM8K / POT / df9c5d15-c741-4ea4-8fd7-49d1f1e90b5a ===
Question:
Carlos is planting a lemon tree. The tree will cost $90 to plant. Each year it will grow 7 lemons, which he can sell for $1.5 each. It costs $3 a year to water and f

## Evaluate thinking

In [5]:
thinking_results = evaluate(examples, model_id="Qwen/Qwen3.5-9B", thinking=True, batch_size=16)

2026-10-29 15:19:37 INFO llm_critic: Starting evaluation: model=Qwen/Qwen3.5-9B examples=48 cache=/sc/home/jan.berndt/better-nli/final/cache/llm_critic.sqlite cache_only=False


Evaluating:   0%|          | 0/48 [00:00<?, ?example/s]

2026-10-29 15:19:38 WARNING llm_critic: UNPARSEABLE MR-GSM8K/original/65889d9b-4418-4d5a-8d1d-06a4372cccc0: missing FINAL line at end of response
The user wants me to identify the first erroneous step in the provided solution.

**Problem Analysis:**
- Bus capacity: 200 people.
- Initial passengers: 20 people.
- Stop 1: 40 people entered.
- Stop 2: 3/4 of the total number of people who entered the bus at the first station entered the bus.
- Stop 3: The number of passengers on the bus doubled.
- Goal: Calculate the total number of people required to fill the remaining spaces on the bus.

**Step-by-Step Verification:**

*   **Step 1:** "The capacity of the bus is 200 people, and it started with 20 people, so there are 200 - 20 = 180 spaces remaining."
    *   Capacity = 200.
    *   Initial = 20.
    *   Remaining spaces = 200 - 20 = 180.
    *   This calculation is correct.

*   **Step 2:** "At the first stop, 40 people entered the bus, so there are now 20 + 40 = 60 people on the bus."
 

In [6]:
display(statistics(thinking_results))
random_answers(thinking_results, examples, seed=42)

,benchmark,task,n,errors,unparseable,coverage,accuracy,error_accuracy,no_error_accuracy,f1,mcc
0,MR-GSM8K,original,4,3,1,0.75,0.50,0.333333,1.000000,1.0,1.0
1,MR-GSM8K,POT,4,1,2,0.50,0.50,1.000000,0.333333,1.0,1.0
2,MR-GSM8K,reversed,4,1,1,0.75,0.50,0.000000,0.666667,0.0,0.0
3,BIG-Bench-Mistake,dyck_languages,4,3,4,0.00,0.00,0.000000,0.000000,NaN,NaN
4,BIG-Bench-Mistake,logical_deduction,4,4,3,0.25,0.25,0.250000,NaN,1.0,0.0
5,BIG-Bench-Mistake,multistep_arithmetic,4,4,3,0.25,0.25,0.250000,NaN,1.0,0.0
6,BIG-Bench-Mistake,tracking_shuffled_objects,4,4,1,0.75,0.75,0.750000,NaN,1.0,0.0
7,BIG-Bench-Mistake,word_sorting,4,4,4,0.00,0.00,0.000000,NaN,NaN,NaN
8,ProcessBench,gsm8k,4,2,3,0.25,0.25,0.000000,0.500000,0.0,0.0
9,ProcessBench,math,4,0,4,0.00,0.00,NaN,0.000000,NaN,NaN



=== MR-GSM8K / original / ffe7fcb1-7651-4385-af1b-43c4e84d6151 ===
Question:
On a certain day, the total cost of filling up 20 helium balloons was $900. Two days later, the cost of filling each balloon was increased by $20. How much did Bentley pay to fill 170 balloons if he went to fill them after the price increase?

Solution steps:
Step 1: On the first day, the cost of filling each balloon was $900 / 20 = $45.
Step 2: Two days later, the cost of filling each balloon increased by $20, so it became $45 + $20 = $65.
Step 3: Bentley filled 170 balloons, so he paid a total of $65 * 170 = $10,550.
Step 4: #### 10550
Step 5: The answer is: 10550

Correct answer: 11050
Ground truth first error step: Step 3
LLM judge's output:
Thinking Process:

1.  **Analyze the Request:** The user wants me to identify the *first* erroneous step in a provided step-by-step solution to a reasoning problem. If the solution is correct, I should output "NONE". The output format must be "FINAL: <integer or NONE>